In [1]:
!pip install --upgrade --force-reinstall \
langchain==0.1.14 \
langchain-core==0.1.37 \
langchain-community==0.0.30 \
faiss-cpu \
groq \
python-dotenv \
pypdf

  Using cached langchain-0.1.14-py3-none-any.whl.metadata (13 kB)
  Using cached langchain_core-0.1.37-py3-none-any.whl.metadata (6.0 kB)
  Using cached langchain_community-0.0.30-py3-none-any.whl.metadata (8.4 kB)
  Using cached faiss_cpu-1.12.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached groq-0.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached pypdf-6.0.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached sqlalchemy-2.0.43-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.6 kB)
  Using cached aiohttp-3.12.15-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached lang

In [2]:
!pip install --upgrade --force-reinstall \
    transformers==4.41.2 \
    sentence-transformers==2.5.1 \
    numpy==1.26.4 \
    packaging==23.2 \
    torch==2.2.2

  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached sentence_transformers-2.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached torch-2.2.2-cp311-cp311-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached regex-2025.7.34-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.19.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x

In [3]:
!pip uninstall -y torchao

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq

In [5]:
PDF_FILE = "/kaggle/input/document/serious_illnesses.pdf"
# PDF_FILE = "/kaggle/input/financial-report/quarterly_financial_report.pdf"
# PDF_FILE = "/kaggle/input/meezan-bank/SOC-ENG-Jan-Jun-2025.pdf"

loader = PyPDFLoader(PDF_FILE)
pages = loader.load() # list of pages are stored here(split by one page)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) #split each page by 500 characters, 50 character overlap from previous chunk to preserve context 
# text_splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20)
# text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
chunks = text_splitter.split_documents(pages) # split accordingly
# print(f"Total chunks: {len(chunks)}")

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") #embedding model initialization(converting text into vectors that captures semantic meaning)

vectorstore = FAISS.from_documents(chunks, embedding_model) #creating a vector database from the chunks and embeddings(Facebook AI Similarity Search)

vectorstore.save_local("faiss_index") #stored locally


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
!pip install langdetect deep-translator

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
from langdetect import detect

def detect_language(text):
    try:
        lang = detect(text)
        return lang
    except:
        return "en"

In [9]:
from transformers import pipeline

intent_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
INTENT_LABELS = ["greeting", "thank_you", "goodbye", "asking about a disease or illness", "unknown"]

2025-08-17 10:57:59.963097: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755428280.290817     204 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755428280.370353     204 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def classify_intent(user_input):
    result = intent_classifier(user_input, INTENT_LABELS)
    print(result["labels"][:3], result["scores"][:3])
    return result["labels"][0]

In [11]:
from typing import List, Dict

chat_history: List[Dict[str, str]] = []

def add_to_history(role, content):
    chat_history.append({"role" : role, "content" : content})

In [12]:
greeting_responses = {
    "en": "Hello! 👋 I'm MediQ, your personalized medical assistant. How can I help you today?",
    "fr": "Bonjour ! 👋 Je suis MediQ, votre assistant médical personnalisé. Comment puis-je vous aider aujourd'hui ?",
    "ur": "ہیلو! 👋 میں میڈی کیو ہوں، آپ کا ذاتی طبی معاون۔ میں آپ کی کس طرح مدد کر سکتا ہوں؟",
    "es": "¡Hola! 👋 Soy MediQ, tu asistente médico personal. ¿En qué puedo ayudarte?",
}

thank_you_responses = {
    "en": "You're very welcome! 😊 Let me know if you have more questions.",
    "fr": "Je vous en prie ! 😊 Faites-moi savoir si vous avez d'autres questions.",
    "ur": "آپ کا خیرمقدم ہے! 😊 اگر کوئی اور سوال ہو تو ضرور بتائیں۔",
    "es": "¡De nada! 😊 Dime si tienes más preguntas.",
}

goodbye_responses = {
    "en": "Goodbye! Take care. 💙",
    "fr": "Au revoir ! Prenez soin de vous. 💙",
    "ur": "خدا حافظ! اپنا خیال رکھیں۔ 💙",
    "es": "¡Adiós! Cuídate. 💙",
}

In [13]:
from deep_translator import GoogleTranslator

def translate_text(text, source='auto', target='en'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception as e:
        print(f"Translation failed: {e}")
        return text  # fallback to original text


In [14]:
def query_bot(user_input):
    lang = detect_language(user_input)
    intent = classify_intent(user_input)

    def get_response_by_lang(template_dict):
        return template_dict.get(lang, template_dict["en"])  # default to English

    if intent == "greeting":
        response = get_response_by_lang(greeting_responses)

    elif intent == "thank_you":
        response = get_response_by_lang(thank_you_responses)

    elif intent == "goodbye":
        response = get_response_by_lang(goodbye_responses)

    elif intent == "asking about a disease or illness":
        if lang != "en":
            translated_input = translate_text(user_input, source=lang, target="en")
            answer_in_english = query_rag(translated_input)
            response = translate_text(answer_in_english, source="en", target=lang)
        else:
            response = query_rag(user_input)

    else:
        response = {
            "en": "I only respond to medical queries.",
            "fr": "Je ne réponds qu'aux questions médicales.",
            "ur": "میں صرف طبی سوالات کے جوابات دیتا ہوں۔",
            "es": "Solo respondo a preguntas médicas.",
        }.get(lang, "I only respond to medical queries.")

    add_to_history("user", user_input)
    add_to_history("assistant", response)
    return response


In [15]:
def rephrase_question_with_history(history, latest_question):
    condensed_chat = "\n".join([
        f"{msg['role'].capitalize()}: {msg['content']}" for msg in history[-4:]
    ])
    rephrase_prompt = f"""Given the conversation below, rewrite the latest user question to make it 
    fully self-contained and unambiguous.

Conversation:
{condensed_chat}

User's latest question: {latest_question}
Rewritten standalone version:"""

    rephrase_response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": rephrase_prompt}]
    )
    return rephrase_response.choices[0].message.content.strip()

In [ ]:
import numpy as np
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
def query_rag(question):
    rewritten_question = rephrase_question_with_history(chat_history, question)

    docs = vectorstore.similarity_search(rewritten_question, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])

    query_embedding = embedding_model.embed_query(rewritten_question)
    top_doc_embedding = embedding_model.embed_query(docs[0].page_content)
    similarity_score = np.dot(query_embedding, top_doc_embedding)

    if similarity_score < 0.3:
        return "I do not have knowledge for this question."

    if not context.strip():
        answer = "I do not have knowledge for this question."
        add_to_history("user", question)
        add_to_history("assistant", answer)
        return answer

    recent_turns = chat_history[-4:]
    conversation_so_far = "\n".join([f"{msg['role'].capitalize()}: {msg['content']}" for msg in recent_turns])

    prompt = f"""Answer the user's question based on the context and conversation history. If the 
    user's question is already specific (e.g., includes disease names like "TB" or "malaria"), 
    prioritize answering it directly even if it seems unrelated to the prior conversation. If the 
    context does **not** mention anything related to the user question, reply with:
    "I do not have knowledge for this question."


Context:
{context}

Conversation so far:
{conversation_so_far}

User: {question}
Assistant:"""

    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.choices[0].message.content.strip()

    add_to_history("user", question)
    add_to_history("assistant", answer)

    if "not provide" in answer.lower() or "does not contain" in answer.lower():
        print("⚠️ The document does not seem to contain the specific answer. Please refer to a medical professional or trusted source.")
    else:
        return answer

**Gradio Implementation**

In [17]:
!pip install gradio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 18.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 63.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 43.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.6 MB/s eta 0:00:00


In [18]:
import gradio as gr

In [19]:
with gr.Blocks() as demo:
    gr.Markdown("## 🩺 MediQ: Your Personalized Medical Assistant ChatBot 💊")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question", placeholder="e.g., what are the symptoms of Malta fever?")
    clear = gr.Button("Clear Chat")

    def handle_submit(user_input, history):
        answer = query_bot(user_input)
        return "", history + [[user_input, answer]]

    msg.submit(handle_submit, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ([], []), outputs=[chatbot, msg])

demo.launch()

/tmp/ipykernel_204/500644057.py:4: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://8941f6afdcce5b4637.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


['greeting', 'unknown', 'thank_you'] [0.9419122934341431, 0.035363636910915375, 0.014391348697245121]
['greeting', 'thank_you', 'unknown'] [0.3811119794845581, 0.2116013616323471, 0.1793476641178131]
['asking about a disease or illness', 'unknown', 'goodbye'] [0.9762845635414124, 0.013913966715335846, 0.0037358198314905167]
['asking about a disease or illness', 'unknown', 'goodbye'] [0.980308473110199, 0.011941259726881981, 0.003199720522388816]
['asking about a disease or illness', 'unknown', 'greeting'] [0.7306042909622192, 0.2397809624671936, 0.01242632232606411]


**Whole RAG Pipeline**

1) Load a document(PDF, CSV)
2) Split document into chunks
3) Convert each chunk into a vector using an embedding model
4) Store each chunk in a vector database for fast similarity search
5) Construct a prompt
6) Make a similarity check for top 3 most similar and relevant chunks and add to the prompt for context.
7) Pass through the LLM and get the answer

**Multilingual Chatbot**

1) User types a message (in any language)

2) Detect the language (e.g., Urdu, French, etc.)

3) Translate the input into English (only if needed)

4) Classify the intent using zero-shot classification (in English for better accuracy)

5) Decide whether to use a greeting response or RAG

6) Translate the answer back to the user’s original language (if needed)

7) Respond in the same language as the user